# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema provided via the following URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
# The mlcroissant 'dataset' object provides access to all metadata and data
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
Inspect available record sets and their fields. All entity references use their Croissant `@id` value for precision.

We'll enumerate available RecordSets (by their `@id`), then inspect their fields and columns.

In [ ]:
# List all available record sets by their @id
print("Available record sets and their @id:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# For this dataset, we expect mostly one main record set for survey responses.

# Display fields and columns for each record set
for record_set in metadata.record_sets:
    print(f"\nRecord Set @id: {record_set.id}, name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        # Some fields link to columns (esp. if source is tabular)
        if hasattr(field, 'columns') and field.columns:
            for column in field.columns:
                print(f"      Column @id: {column.id}, name: {column.name}, dataType: {column.data_type}")

## 3. Data Extraction
Load the data from the main record set (using its `@id`) as a DataFrame. 

All subsequent processing will reference the record set and its fields by their Croissant `@id` values, such as `'cr:RecordSet-Responses'`, `'cr:Field-age'`, etc.

Let's first define the main record set and its fields for extraction.

In [ ]:
# Step 1: Collect all record set @ids
record_sets = [rs.id for rs in metadata.record_sets]
print(f"Record sets: {record_sets}")

# For this dataset, the main survey data is typically the first record set
record_set_id = record_sets[0]

# Step 2: Load each record set into a DataFrame
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Columns in main record set (@id: {record_set_id}):\n{dataframes[record_set_id].columns.tolist()}")
# View first few rows
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We now select numeric fields and perform basic EDA: filtering, normalization, and grouping. Field names will be referenced by their `@id`.

Common numeric fields in mental health surveys include GAD-7, PHQ-9, or PSQ scores. We'll attempt to locate such a field in the record set, referencing its `@id`.

In [ ]:
# Inspect numeric fields in the main record set
numeric_fields = []
for field in metadata.record_set(record_set_id).fields:
    if field.data_type in ('Number', 'Float', 'Integer'):
        print(f"@id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        numeric_fields.append(field.id)

# Select a numeric field by Croissant @id. Choose GAD-7 score, e.g. 'cr:Field-gad7_score', if present
numeric_field_id = numeric_fields[0] if numeric_fields else None
print(f"Using numeric field: {numeric_field_id}")

df = dataframes[record_set_id]

if numeric_field_id:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another categorical field (e.g., 'cr:Field-gender'), if present
    # We'll search for a field with 'gender' in its name
    group_fields = [field for field in df.columns if 'gender' in field.lower()]
    group_field = group_fields[0] if group_fields else None
    print(f"Grouping by field: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data (mean {numeric_field_id}) by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Let's visualize a histogram of the main numeric field and, if available, compare distributions across genders.

If additional visualization fields are present in the schema, you may reference their `@id` in further analyses.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field (e.g., gender) exists, plot distributions per group
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook showcased how to explore the FAIR² Mental Health Survey dataset from Kilifi, Kenya using the `mlcroissant` library, referencing all entities via their Croissant `@id` fields. 

- We reviewed the metadata and record set structures
- Loaded data into pandas DataFrames
- Performed basic filtering, normalization, and group statistics
- Visualized the results

This approach ensures replicability and schema-aligned data analysis for FAIR AI-Ready datasets.